In [21]:
import glob
import sqlite3
import pandas as pd

files = sorted(glob.glob("../data/raw/2023/*.csv"))
print(len(files))

def read_month(path):
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        print("cp1252:", path)
        return pd.read_csv(path, encoding="cp1252")

df = pd.concat([read_month(f) for f in files], ignore_index=True)

12
cp1252: ../data/raw/2023/Bike share ridership 2023-04.csv
cp1252: ../data/raw/2023/Bike share ridership 2023-05.csv
cp1252: ../data/raw/2023/Bike share ridership 2023-06.csv
cp1252: ../data/raw/2023/Bike share ridership 2023-07.csv
cp1252: ../data/raw/2023/Bike share ridership 2023-08.csv
cp1252: ../data/raw/2023/Bike share ridership 2023-09.csv
cp1252: ../data/raw/2023/Bike share ridership 2023-10.csv
cp1252: ../data/raw/2023/Bike share ridership 2023-11.csv


In [37]:
df.head()

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model,is_ebike
0,20148784,840,7022,2023-01-01 00:00:00,Simcoe St / Queen St W,7703,2023-01-01 00:14:00,NaN,1770,Casual Member,<NA>,<NA>
1,20148785,722,7399,2023-01-01 00:01:00,Lower Jarvis / Queens Quay E,7533,2023-01-01 00:13:00,Housey St / Dan Leckie Way,6400,Casual Member,<NA>,<NA>
2,20148786,1054,7269,2023-01-01 00:02:00,Toronto Eaton Centre (Yonge St),7076,2023-01-01 00:20:00,York St / Queens Quay W,4068,Annual Member,<NA>,<NA>
3,20148790,1329,7721,2023-01-01 00:04:00,NaN,7685,2023-01-01 00:26:00,NaN,427,Casual Member,<NA>,<NA>
4,20148791,1291,7721,2023-01-01 00:04:00,NaN,7685,2023-01-01 00:26:00,NaN,1840,Casual Member,<NA>,<NA>


In [2]:
df.describe()

,Trip Id,Trip Duration,Start Station Id,End Station Id,Bike Id
count,5.713141e+06,5.713141e+06,5.713141e+06,5.710197e+06,5.713141e+06
mean,2.342688e+07,1.079395e+03,7.304815e+03,7.300489e+03,3.914233e+03
std,1.888400e+06,8.797974e+03,2.298876e+02,2.306553e+02,2.193457e+03
min,2.014878e+07,0.000000e+00,7.000000e+03,7.000000e+03,1.400000e+01
25%,2.179640e+07,4.210000e+02,7.102000e+03,7.099000e+03,2.074000e+03
50%,2.342814e+07,6.870000e+02,7.261000e+03,7.257000e+03,3.973000e+03
75%,2.506422e+07,1.120000e+03,7.467000e+03,7.463000e+03,5.758000e+03
max,2.668274e+07,3.980837e+06,7.918000e+03,7.918000e+03,8.724000e+03


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5713141 entries, 0 to 5713140
Data columns (total 10 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Trip Id             int64  
 1   Trip  Duration      int64  
 2   Start Station Id    int64  
 3   Start Time          str    
 4   Start Station Name  str    
 5   End Station Id      float64
 6   End Time            str    
 7   End Station Name    str    
 8   Bike Id             int64  
 9   User Type           str    
dtypes: float64(1), int64(4), str(5)
memory usage: 942.6 MB


In [22]:
df["Bike_Model"] = pd.NA
df["is_ebike"] = pd.NA

In [25]:
df.columns

Index(['Trip_Id', 'Trip_Duration', 'Start_Station_Id', 'Start_Time',
       'Start_Station_Name', 'End_Station_Id', 'End_Time', 'End_Station_Name',
       'Bike_Id', 'User_Type', 'Bike_Model', 'is_ebike'],
      dtype='str')

In [43]:
df.isna().sum()

Trip_Id                     0
Trip_Duration               0
Start_Station_Id            0
Start_Time                  0
Start_Station_Name     591447
End_Station_Id           1659
End_Time                    0
End_Station_Name       593694
Bike_Id                     0
User_Type             5681771
Bike_Model            5681771
is_ebike              5681771
dtype: int64

In [24]:
df.columns = df.columns.str.strip().str.replace(r"\s+", "_", regex= True)

In [27]:
df.isna().sum()

Trip_Id                     0
Trip_Duration               0
Start_Station_Id            0
Start_Time                  0
Start_Station_Name     595075
End_Station_Id           2944
End_Time                    0
End_Station_Name       598563
Bike_Id                     0
User_Type                   0
Bike_Model            5713141
is_ebike              5713141
dtype: int64

In [28]:
df["Trip_Duration"].quantile([0.99,0.999,0.9999])

0.9900      5347.000
0.9990     27162.000
0.9999    256247.744
Name: Trip_Duration, dtype: float64

In [29]:
df = df[df["Trip_Duration"] <= 12875.00]

In [47]:
df.dtypes

Trip_Id                        int64
Trip_Duration                  int64
Start_Station_Id               int64
Start_Time            datetime64[us]
Start_Station_Name               str
End_Station_Id                 Int64
End_Time              datetime64[us]
End_Station_Name                 str
Bike_Id                        int64
User_Type                     string
Bike_Model                    string
is_ebike                     boolean
dtype: object

In [31]:
df["Start_Time"] = pd.to_datetime(df["Start_Time"])
df["End_Time"] = pd.to_datetime(df["End_Time"])

In [33]:
df["End_Station_Id"] = df["End_Station_Id"].astype("Int64")

In [46]:
df["User_Type"] = df["User_Type"].astype("string")
df["Bike_Model"] = df["Bike_Model"].astype("string")
df["is_ebike"] = df["is_ebike"].astype("boolean")

In [34]:
(df["Trip_Id"].duplicated()).sum()

np.int64(0)

In [36]:
df["User_Type"].value_counts()

User_Type
Casual Member    5336464
Annual Member     364151
Name: count, dtype: int64

The official reports contradict the open data’s labels, so I excluded them.

Sources:

https://www.toronto.ca/legdocs/mmis/2024/pa/bgrd/backgroundfile-251202.pdf
https://www.toronto.ca/legdocs/mmis/2023/pa/bgrd/backgroundfile-240804.pdf
https://urbantoronto.ca/forum/threads/general-cycling-issues-is-toronto-bike-friendly.4842/page-291#post-2287665 


In [38]:
df["User_Type"] = pd.NA

In [39]:
(df["Start_Station_Id"] == df["End_Station_Id"]).sum()

np.int64(181516)

In [40]:
round_trips = df[df["Start_Station_Id"] == df["End_Station_Id"]]

In [41]:
round_trips["Trip_Duration"].describe()

count    181516.000000
mean       1962.133211
std        2081.215968
min           0.000000
25%         460.000000
50%        1328.000000
75%        2628.000000
max       12874.000000
Name: Trip_Duration, dtype: float64

In [42]:
is_redock =(df["Start_Station_Id"] == df["End_Station_Id"]) & (df["Trip_Duration"] < 120)
df = df[~is_redock]

In [1]:
conn = sqlite3.connect("../data/processed/bikeshare.db")
df.to_sql("trips", conn, if_exists="append", index=False)
conn.close()

NameError: name 'sqlite3' is not defined